# ERA5 Land Data
ERA5 data extraction notebook.


In [ ]:

import ee
import os
import pandas as pd
import matplotlib.pyplot as plt
import time
import datetime
print('Libraries loaded.')


In [ ]:
# Authenticate and initialize ----------------------------------------------------------------------
ee.Authenticate()
ee.Initialize(project='fire-seasons')

In [ ]:
# Imports and configurations -----------------------------------------------------------------------

YEARS = range(2003, 2026)
SCALE = 11132
ERA5_COLLECTION = 'ECMWF/ERA5_LAND/DAILY_AGGR'
BANDS = ['temperature_2m', 'total_precipitation_sum']
RUN_ID = datetime.datetime.now().strftime('%Y%m%d_%H%M')

# Output paths and directories -----------------------------------------------------------------------

BASE_OUT_DIR = r'C:\Users\ibekar\Documents\GitProjects\TGPF'  # Windows
# BASE_OUT_DIR = '/Users/ibekar/Github/TGPF'                  # Mac
DRIVE_FOLDER = "ERA5_raw"

save_dir = os.path.join(BASE_OUT_DIR, 'inputs', 'raw_data', 'era5')
print(save_dir)

In [ ]:
# LOAD MEDITERRANEAN BASIN ECOREGIONS --------------------------------------------------------------
med_bbox = ee.Geometry.BBox(-10, 28, 42, 48)

ecoregions_med = ee.FeatureCollection("RESOLVE/ECOREGIONS/2017").filterBounds(med_bbox)

n_eco    = ecoregions_med.size().getInfo()
eco_list = ecoregions_med.select(['ECO_ID', 'ECO_NAME', 'BIOME_NUM', 'BIOME_NAME']).getInfo()

print(f'Number of ecoregions intersecting Mediterranean bounding box: {n_eco}')
for f in eco_list['features']:
    p = f['properties']
    print(p['ECO_ID'], '|', p['ECO_NAME'], '|', p['BIOME_NAME'])

# BUILD ECO RECORDS --------------------------------------------------------------------------------
eco_records = []
for f in eco_list['features']:
    p = f['properties']
    eco_records.append({
        'eco_id'    : p['ECO_ID'],
        'eco_name'  : p['ECO_NAME'],
        'biome_num' : p['BIOME_NUM'],
        'biome_name': p['BIOME_NAME'],
        'geometry'  : ee.Geometry(f['geometry'])
    })

print(f'Built {len(eco_records)} ecoregion records.')

# SUBSETTING (set to None to disable) --------------------------------------------------------------
TEST_N   = None
TEST_IDS = None

eco_run = eco_records

if TEST_IDS is not None:
    eco_run = [e for e in eco_run if e['eco_id'] in TEST_IDS]
    print(f'Subsetting to {len(eco_run)} ecoregions by ID: {TEST_IDS}')

if TEST_N is not None:
    eco_run = eco_run[:TEST_N]
    print(f'Subsetting to first {TEST_N} ecoregions.')

print(f'Running on {len(eco_run)} / {len(eco_records)} ecoregions.')

In [ ]:
def extract_era5_for_ecoregion(eco_feature):
    '''Extract ERA5 data for a single eco-region feature.'''
    eco_id = eco_feature['eco_id']
    eco_name = eco_feature['eco_name']
    geometry = eco_feature['geometry']
    geometry_simplified = geometry.simplify(500)  # ← simplify once here

    # Filter ERA5 collection by date and geometry
    era5 = (ee.ImageCollection(ERA5_COLLECTION)
            .filterDate(f'{YEARS[0]}-01-01', f'{YEARS[-1]}-12-31')
            .select(BANDS))
    
    # Define a per image mapping function

    def image_to_feature(image):
        stats = image.reduceRegion(
            reducer = ee.Reducer.mean(),
            geometry = geometry_simplified, # Simplify geometry to speed up processing
            scale = SCALE, # ERA5-Land native resolution ~0.1°
            maxPixels = 1e9,
            bestEffort = True # Allow processing of large geometries
        )
        return ee.Feature(None, {
            "eco_id" : eco_id,
            "date" : image.date().format('YYYY-MM-dd'),
            "eco_name" : eco_name,
            "temperature_K" : stats.get('temperature_2m'),
            "precip_m" : stats.get('total_precipitation_sum')
        })
    daily_fc = ee.FeatureCollection(era5.map(image_to_feature))
    return daily_fc

In [ ]:
n_submitted = 0

for e in eco_run:
    task_desc = f'ERA5_{RUN_ID}_eco_{e["eco_id"]}'
    file_name = f'ERA5_eco_{e["eco_id"]}_{e["eco_name"].replace(" ", "_")}'
    print(f'Submitting {task_desc}...')

    daily_fc = extract_era5_for_ecoregion(e)

    task = ee.batch.Export.table.toDrive(
        collection    = daily_fc,
        description   = task_desc,
        folder        = DRIVE_FOLDER,
        fileNamePrefix= file_name,
        fileFormat    = 'CSV',
        selectors     = ['eco_id', 'eco_name', 'date', 'temperature_K', 'precip_m']
    )
    task.start()
    print(f'  → Submitted.')
    n_submitted += 1

print(f'All {n_submitted} tasks submitted.')
print('Monitor at: https://code.earthengine.google.com/tasks')

In [ ]:
print('Monitoring tasks...')
while True:
    tasks     = ee.data.getTaskList()
    running   = sum(1 for t in tasks if t['state'] == 'RUNNING')
    completed = sum(1 for t in tasks if t['state'] == 'COMPLETED')
    failed    = sum(1 for t in tasks if t['state'] == 'FAILED')
    ready     = sum(1 for t in tasks if t['state'] == 'READY')

    print(f'  READY: {ready} | RUNNING: {running} | COMPLETED: {completed} | FAILED: {failed}')

    if running == 0 and ready == 0:
        print('All tasks finished.')
        break

    time.sleep(120)

In [ ]:
PLOT_ECO_ID = 701  # change this to inspect any ecoregion

plot_path = os.path.join(save_dir, f'ERA5_eco_{PLOT_ECO_ID}.csv')
out = pd.read_csv(plot_path)

out["date"] = pd.to_datetime(out["date"])


plt.figure(figsize=(12, 6))
plt.suptitle(f'ERA5 Daily Data for Ecoregion {out['eco_name'].iloc[0]}', fontsize=16)
plt.subplot(2, 1, 1)
plt.plot(out["date"], out["precip_m"])
plt.title("Daily Precipitation (m)")
plt.subplot(2, 1, 2)
plt.plot(out["date"], out["temperature_K"])
plt.title("Daily Temperature (K)")
plt.xlabel("Date")
plt.ylabel("Value")
plt.tight_layout()
plt.show()
